# Mechanistic in Silico Simulation Results Notebook

## Instructions

[Comprehensive documentation here](Docs/simglucose.md) detailing how to run Trio oref algorithm variants through simglucose to conduct mechanistic in silico simulations.

### Run One Virtual Person

```
python3 simglucose/run_sim.py -u <virtual_patient> -a <alg_name> -d <days> -scen <meal_scenario_path> -fn <results_file>
```
* `virtual_patient`: Name of virtual patient. Valid patient names are age group followed by three digits. Age groups = [child, adolescent, adult]. Valid digits = [001, 002, 003, 004, 005, 006, 007, 008, 009, 010]
    * eg. adolescent002 or adult010
* `alg_name`: Optional argument. Name of the Javascript oref algorithm variant to run instead of the Swift algorithm. Defaults to `"swift"` if no argument given. Possible choices:
    * `jsbug`: Original Javascript implementation.
    * `js`: Javascript implementation of bug-free Swift oref algorithm.
    * `swift`: Swift implementation of oref algorithm.
* `days`: Number of days simulation runs (must be whole number)
* `meal_scenario_path`: Optional argument. Filepath to precomputed .npy file containing meal scenario.
* `results_file`: Filepath to csv file where outputs will be written to.

### Run All Virtual People
Run the following commands to execute this [script](./Scripts/run_simglucose.sh) which simulates all 30 virtual persons. There is the option to run simulations in parallel in independent processes. Set `PARALLELISM` to the number of processes you want to run concurrently. 

```shell
chmod +x Scripts/run_simglucose.sh # if have not run script yet
./Scripts/run_simglucose.sh
```

## Glucose Metric Analysis

In [4]:
import numpy as np
import pandas as pd
from typing import Literal

In [7]:
def patient_names(age: Literal["adult", "adolescent", "child"]):
    code_names = [f"{age}{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
    age_cap = f"{age[0].upper()}{age[1:]}"
    display_names = [f"{age_cap} {n}" for n in range(1,11)]
    return code_names, display_names

In [13]:
children, children_display = patient_names("child")
adolescents, adolescents_display = patient_names("adolescent")
adults, adults_display = patient_names("adult")
all_users = children + adolescents + adults
all_users_display = children_display + adolescents_display + adults_display
results_folder = "simglucoseResults"

In [14]:
def glycemia_risk_index(tar, tbr, tvar, tvbr):
    gri = 3 * tvbr + 2.4 * tbr + 1.6 * tvar + 0.8 * tar
    return 100.0 if gri > 100.0 else gri

def glucose_stats(glucose):
    low_bound = 70
    v_low_bound = 54
    v_high_bound = 250
    high_bound = 180
    glucose = np.array(glucose)

    tar = 100 * np.average(glucose > high_bound)
    tvar = 100 * np.average(glucose > v_high_bound)
    tbr = 100 * np.average(glucose < low_bound)
    tvbr = 100 * np.average(glucose < v_low_bound)
    tir = 100 - tar - tbr

    gri = glycemia_risk_index(tar, tbr, tvar, tvbr)

    return tir, tar, tbr, tvar, tvbr, gri

In [44]:
def user_results(user:str, alg:str):
    path = f"{results_folder}/{user}/{alg}.csv"
    glucose = pd.read_csv(path)['CGM'].to_list()
    return glucose_stats(glucose)

def table_round(n, pct:bool):
    end = '%' if pct else ''
    return f"{round(n, 1)}{end}"
    
def age_group_results(group:list, user_display:list, alg:str):
    results = {"User":[], "TIR":[], "TAR":[], "TBR":[], "TVAR":[], 
                    "TVBR":[], "GRI":[]}
    raw_results = {"TIR":[], "TAR":[], "TBR":[], "TVAR":[], "TVBR":[], "GRI":[]}
    
    for i in range(len(group)):
        tir, tar, tbr, tvar, tvbr, gri = user_results(group[i], alg)
        results["User"].append(user_display[i])
        results["TIR"].append(table_round(tir, True))
        raw_results["TIR"].append(tir)
        results["TAR"].append(table_round(tar, True))
        raw_results["TAR"].append(tar)
        results["TBR"].append(table_round(tbr, True))
        raw_results["TBR"].append(tbr)
        results["TVAR"].append(table_round(tvar, True))
        raw_results["TVAR"].append(tvar)
        results["TVBR"].append(table_round(tvbr, True))
        raw_results["TVBR"].append(tvbr)
        results["GRI"].append(table_round(gri, False))
        raw_results["GRI"].append(gri)

    for category in results.keys():
        if category == "User": 
            results["User"].append("Mean")
            results["User"].append("Standard Deviation")
        else:
            avg = np.average(raw_results[category]).item()
            sd = np.std(raw_results[category]).item()
            show_pct = category != "GRI"
            results[category].extend([table_round(avg, show_pct), table_round(sd, show_pct)])

    return results

In [45]:
print('Trio Swift Implemention Simulation Results on Children')
df = pd.DataFrame.from_dict(age_group_results(children, children_display, 'swift'))
display(df.style.hide())

print('Trio Javascript Implemention Simulation Results on Children')
df = pd.DataFrame.from_dict(age_group_results(children, children_display, 'jsbug'))
display(df.style.hide())

Trio Swift Implemention Simulation Results on Children


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Child 1,78.9%,15.8%,5.3%,4.2%,1.1%,35.3
Child 2,78.8%,16.8%,4.4%,0.1%,0.8%,26.7
Child 3,67.4%,26.7%,5.8%,8.7%,0.8%,51.6
Child 4,84.4%,13.1%,2.4%,0.5%,0.3%,18.1
Child 5,89.4%,6.1%,4.4%,0.0%,0.7%,17.6
Child 6,77.6%,20.3%,2.1%,4.9%,0.1%,29.5
Child 7,92.5%,5.2%,2.3%,0.1%,0.2%,10.5
Child 8,46.4%,44.1%,9.4%,32.0%,4.7%,100.0
Child 9,89.2%,9.2%,1.6%,0.2%,0.1%,12.0
Child 10,71.0%,27.0%,2.0%,12.6%,0.3%,47.6


Trio Javascript Implemention Simulation Results on Children


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Child 1,78.7%,15.6%,5.7%,4.2%,1.6%,37.6
Child 2,80.3%,15.1%,4.7%,0.1%,0.8%,25.8
Child 3,66.6%,26.6%,6.8%,8.6%,0.5%,53.0
Child 4,83.7%,12.8%,3.5%,0.9%,0.5%,21.6
Child 5,88.8%,6.4%,4.8%,0.0%,0.9%,19.3
Child 6,77.3%,20.3%,2.4%,4.9%,0.3%,30.7
Child 7,94.1%,4.1%,1.8%,0.0%,0.3%,8.7
Child 8,46.9%,44.0%,9.1%,32.2%,4.7%,100.0
Child 9,88.7%,9.4%,1.9%,0.2%,0.1%,12.9
Child 10,71.5%,26.9%,1.6%,12.1%,0.1%,45.2


In [46]:
print('Trio Swift Implemention Simulation Results on Adolescents')
df = pd.DataFrame.from_dict(age_group_results(adolescents, adolescents_display, 'swift'))
display(df.style.hide())

print('Trio Javascript Implemention Simulation Results on Adolescents')
df = pd.DataFrame.from_dict(age_group_results(adolescents, adolescents_display, 'jsbug'))
display(df.style.hide())

Trio Swift Implemention Simulation Results on Adolescents


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Adolescent 1,99.3%,0.1%,0.7%,0.0%,0.0%,1.7
Adolescent 2,63.4%,34.2%,2.5%,4.6%,0.2%,41.2
Adolescent 3,92.3%,4.0%,3.7%,0.0%,0.6%,13.9
Adolescent 4,87.1%,6.7%,6.2%,0.0%,1.0%,23.3
Adolescent 5,77.7%,17.9%,4.4%,2.0%,0.9%,30.9
Adolescent 6,93.2%,5.6%,1.1%,0.0%,0.2%,7.9
Adolescent 7,69.6%,27.3%,3.1%,5.9%,0.5%,40.2
Adolescent 8,72.3%,24.8%,2.9%,3.5%,0.0%,32.5
Adolescent 9,94.6%,4.1%,1.2%,0.0%,0.0%,6.4
Adolescent 10,91.3%,4.1%,4.5%,0.0%,0.6%,16.0


Trio Javascript Implemention Simulation Results on Adolescents


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Adolescent 1,99.2%,0.1%,0.8%,0.0%,0.0%,1.9
Adolescent 2,61.7%,35.8%,2.5%,4.5%,0.3%,42.7
Adolescent 3,93.0%,3.9%,3.2%,0.0%,0.1%,11.1
Adolescent 4,88.0%,6.8%,5.2%,0.0%,0.8%,20.3
Adolescent 5,77.3%,18.0%,4.7%,1.8%,0.8%,30.8
Adolescent 6,92.9%,5.8%,1.3%,0.0%,0.2%,8.4
Adolescent 7,70.1%,27.1%,2.8%,6.0%,0.5%,39.3
Adolescent 8,72.8%,25.0%,2.2%,3.4%,0.0%,30.8
Adolescent 9,94.6%,3.9%,1.5%,0.0%,0.0%,6.7
Adolescent 10,91.5%,4.0%,4.5%,0.0%,0.8%,16.4


In [47]:
print('Trio Swift Implemention Simulation Results on Adults')
df = pd.DataFrame.from_dict(age_group_results(adults, adults_display, 'swift'))
display(df.style.hide())

print('Trio Javascript Implemention Simulation Results on Adults')
df = pd.DataFrame.from_dict(age_group_results(adults, adults_display, 'jsbug'))
display(df.style.hide())

Trio Swift Implemention Simulation Results on Adults


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Adult 1,92.2%,2.7%,5.1%,0.0%,0.5%,16.0
Adult 2,97.8%,0.4%,1.8%,0.0%,0.1%,4.8
Adult 3,95.9%,0.2%,3.9%,0.0%,0.9%,12.3
Adult 4,92.0%,6.3%,1.6%,0.1%,0.1%,9.6
Adult 5,97.6%,1.1%,1.4%,0.0%,0.0%,4.1
Adult 6,87.4%,2.8%,9.9%,0.0%,1.2%,29.4
Adult 7,94.5%,1.0%,4.5%,0.0%,0.6%,13.2
Adult 8,97.5%,0.1%,2.4%,0.0%,0.3%,6.6
Adult 9,82.4%,8.3%,9.3%,0.2%,1.8%,34.5
Adult 10,94.1%,1.6%,4.4%,0.0%,0.3%,12.6


Trio Javascript Implemention Simulation Results on Adults


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Adult 1,91.9%,2.8%,5.3%,0.0%,0.4%,16.2
Adult 2,97.4%,0.4%,2.2%,0.0%,0.1%,5.8
Adult 3,95.6%,0.2%,4.2%,0.0%,0.6%,12.1
Adult 4,92.5%,5.9%,1.5%,0.1%,0.2%,9.1
Adult 5,97.7%,1.0%,1.3%,0.0%,0.1%,4.5
Adult 6,87.9%,2.5%,9.5%,0.0%,1.7%,30.0
Adult 7,94.4%,1.0%,4.5%,0.0%,0.6%,13.4
Adult 8,97.4%,0.1%,2.5%,0.0%,0.3%,6.9
Adult 9,82.0%,8.8%,9.2%,0.2%,1.8%,34.9
Adult 10,93.3%,1.4%,5.3%,0.0%,0.5%,15.2


In [70]:
print('Trio Swift Implemention Simulation Per User Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, all_users_display, 'swift'))
display(df.style.hide())


print('Trio Javascript Implemention Simulation Per User Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, all_users_display, 'jsbug'))
display(df.style.hide())

Trio Swift Implemention Simulation Per User Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Child 1,78.9%,15.8%,5.3%,4.2%,1.1%,35.3
Child 2,78.8%,16.8%,4.4%,0.1%,0.8%,26.7
Child 3,67.4%,26.7%,5.8%,8.7%,0.8%,51.6
Child 4,84.4%,13.1%,2.4%,0.5%,0.3%,18.1
Child 5,89.4%,6.1%,4.4%,0.0%,0.7%,17.6
Child 6,77.6%,20.3%,2.1%,4.9%,0.1%,29.5
Child 7,92.5%,5.2%,2.3%,0.1%,0.2%,10.5
Child 8,46.4%,44.1%,9.4%,32.0%,4.7%,100.0
Child 9,89.2%,9.2%,1.6%,0.2%,0.1%,12.0
Child 10,71.0%,27.0%,2.0%,12.6%,0.3%,47.6


Trio Javascript Implemention Simulation Per User Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
Child 1,78.7%,15.6%,5.7%,4.2%,1.6%,37.6
Child 2,80.3%,15.1%,4.7%,0.1%,0.8%,25.8
Child 3,66.6%,26.6%,6.8%,8.6%,0.5%,53.0
Child 4,83.7%,12.8%,3.5%,0.9%,0.5%,21.6
Child 5,88.8%,6.4%,4.8%,0.0%,0.9%,19.3
Child 6,77.3%,20.3%,2.4%,4.9%,0.3%,30.7
Child 7,94.1%,4.1%,1.8%,0.0%,0.3%,8.7
Child 8,46.9%,44.0%,9.1%,32.2%,4.7%,100.0
Child 9,88.7%,9.4%,1.9%,0.2%,0.1%,12.9
Child 10,71.5%,26.9%,1.6%,12.1%,0.1%,45.2


## Parkes Error Analysis

In [49]:
from ParkesErrorGrid.parkes_error import ParkesError

In [50]:
def user_parkes_grid(user, user_display=None, gen_plot=False):

    if gen_plot and not user_display:
        raise Exception(f"Must give value for user_display if gen_plot=True.")
    
    ref_trace = pd.read_csv(f"simglucoseResults/{user}/swift.csv")["CGM"].to_list()
    pred_trace = pd.read_csv(f"simglucoseResults/{user}/jsbug.csv")["CGM"].to_list()
    grid = ParkesError(ref_trace, pred_trace)
    if gen_plot:
        grid.plot(user_display, "Swift Algorithm Glucose", 
                "Original Javascript Algorithm Glucose", size=1, 
                save_fig_path=f"simglucoseResults/{user}/parkes_error.png")
        
    grid_flip = ParkesError(pred_trace, ref_trace)
    if gen_plot:
        grid_flip.plot(user_display, "Original Javascript Algorithm Glucose",
                "Swift Algorithm Glucose", size=1, 
                save_fig_path=f"simglucoseResults/{user}/parkes_error_flipped.png")

    return grid.zone_count(), grid_flip.zone_count()


In [64]:
swift_ref_users = {}
js_ref_users = {}

for user, user_display in zip(all_users, all_users_display):
    swift_ref, js_ref = user_parkes_grid(user)
    n = sum(swift_ref.values())
    swift_ref_users[user_display] = [table_round(100*count/n, True) for count in swift_ref.values()]
    js_ref_users[user_display] = [table_round(100*count/n, True) for count in js_ref.values()]

In [65]:
cols = ['Zone A', 'Zone B', 'Zone C', 'Zone D', 'Zone E']

print("Parkes error analysis results with Swift implementation being reference algorithm.")
df = pd.DataFrame.from_dict(swift_ref_users, orient='index', columns=cols)
display(df)

Parkes error analysis results with Swift implementation being reference algorithm.


,Zone A,Zone B,Zone C,Zone D,Zone E
Child 1,97.6%,2.4%,0.0%,0.0%,0.0%
Child 2,100.0%,0.0%,0.0%,0.0%,0.0%
Child 3,100.0%,0.0%,0.0%,0.0%,0.0%
Child 4,92.0%,7.8%,0.2%,0.0%,0.0%
Child 5,99.8%,0.2%,0.0%,0.0%,0.0%
Child 6,100.0%,0.0%,0.0%,0.0%,0.0%
Child 7,100.0%,0.0%,0.0%,0.0%,0.0%
Child 8,99.9%,0.1%,0.0%,0.0%,0.0%
Child 9,99.4%,0.6%,0.0%,0.0%,0.0%
Child 10,99.9%,0.1%,0.0%,0.0%,0.0%


In [66]:
print("Parkes error analysis results with Javascript implementation being reference algorithm.")
df = pd.DataFrame.from_dict(js_ref_users, orient='index', columns=cols)
display(df)

Parkes error analysis results with Javascript implementation being reference algorithm.


,Zone A,Zone B,Zone C,Zone D,Zone E
Child 1,98.9%,1.1%,0.0%,0.0%,0.0%
Child 2,100.0%,0.0%,0.0%,0.0%,0.0%
Child 3,99.9%,0.1%,0.0%,0.0%,0.0%
Child 4,92.6%,7.4%,0.0%,0.0%,0.0%
Child 5,100.0%,0.0%,0.0%,0.0%,0.0%
Child 6,100.0%,0.0%,0.0%,0.0%,0.0%
Child 7,100.0%,0.0%,0.0%,0.0%,0.0%
Child 8,99.8%,0.2%,0.0%,0.0%,0.0%
Child 9,99.5%,0.5%,0.0%,0.0%,0.0%
Child 10,100.0%,0.0%,0.0%,0.0%,0.0%
